# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR\(^2\) dataset using the `mlcroissant` library. The dataset focuses on protein abundance, modifications, and sequence analysis from human extracellular vesicles isolated from mast cells using mass spectrometry.

### Dataset Source
The dataset is described by a Croissant schema at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema provides a structure describing which record sets and fields are accessible. Let's inspect these components.

In [ ]:
# List available record sets and inspect their fields by @id
record_sets = metadata.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for recset in record_sets:
    print(f"Record Set Name: {recset.name}")
    print(f"  @id: {recset.id}")
    print(f"  Description: {recset.description}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s listed above.

We'll load each available record set by referencing its `@id`.

In [ ]:
# Gather all record_set IDs
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
# For demonstration, print names and load each record set
for recset in dataset.metadata.record_sets:
    print(f"Loading records for Record Set: {recset.name} (@id: {recset.id})")
    records = list(dataset.records(record_set=recset.id))
    dataframes[recset.id] = pd.DataFrame(records)
    if len(records) > 0:
        print(f"  Loaded {len(records)} records, columns: {dataframes[recset.id].columns.tolist()}")
    else:
        print(f"  No records available.")
    print()
# Let's pick the primary data table for further analysis.
# For this dataset, the main data is typically under the largest or most detailed record set.
main_recset = None
for recset in dataset.metadata.record_sets:
    if dataframes[recset.id].shape[0] > 0:
        main_recset = recset
        break
if not main_recset:
    raise RuntimeError('No populated record sets found.')

main_df = dataframes[main_recset.id]
print(f"Analyzing primary record set: {main_recset.name} (@id: {main_recset.id})")
print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
We can now explore, filter, and transform the data in this record set. We'll:
- Select a numeric field for analysis
- Filter rows by a threshold
- Normalize numeric values
- Optionally, group by a categorical field

All field selections are referenced by their `@id`.

In [ ]:
# Identify a numeric field (by @id) to analyze.
import numpy as np

# Let's try to heuristically choose a numeric field; otherwise, manually select by NAME or ID.
numeric_candidates = [field for field in main_recset.fields if field.data_type in ('Float', 'Integer', 'Number')]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0].id
    numeric_field_name = numeric_candidates[0].name
else:
    # Fallback: try to pick a column that seems numeric from DataFrame
    numeric_field_id = main_df.select_dtypes(include=[np.number]).columns[0]
    numeric_field_name = numeric_field_id

print(f"Using numeric field: {numeric_field_name} (@id: {numeric_field_id})\n")

# Set a threshold value (choose an appropriate value based on the data).
if main_df[numeric_field_id].dtype in [int, float, np.float64, np.int64]:
    threshold = float(main_df[numeric_field_id].mean())  # Use mean as a demo threshold
else:
    threshold = 10.0

filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:\n{filtered_df.head()}")

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Optionally group by another field (categorical)
group_candidates = [field for field in main_recset.fields if field.data_type not in ('Float', 'Integer', 'Number')]
if group_candidates:
    group_field_id = group_candidates[0].id
    group_field_name = group_candidates[0].name
    print(f"\nGrouping by field: {group_field_name} (@id: {group_field_id})")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(grouped_df.head())
else:
    print("No suitable group-by field detected.")

## 5. Visualization
Visualize data distributions or relationships.

We'll show a histogram of the selected numeric field and, if possible, a bar chart for grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of numeric field (filtered)
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True, color='steelblue')
plt.title(f"Distribution of {numeric_field_name} (>{threshold:.2f})")
plt.xlabel(numeric_field_name)
plt.ylabel('Count')
plt.show()

# If grouped data available, plot means
if 'grouped_df' in locals():
    grouped_df.plot(kind='bar', legend=False, figsize=(10, 5))
    plt.title(f"Mean {numeric_field_name} by {group_field_name}")
    plt.ylabel(f"Mean {numeric_field_name}")
    plt.xlabel(group_field_name)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- This notebook demonstrated loading a FAIR\(^2\) Croissant dataset using the `mlcroissant` library referenced strictly by `@id`.
- We identified available record sets and fields, extracted the primary data table, and performed filtering and normalization referencing `@id`s.
- Simple EDA and visualization steps illustrated data trends.

For further exploration, refer to the dataset's Croissant schema for more advanced record linkages, or inspect additional record sets or custom data fields.